# שיעור 05 - RAG סוכני


## התקנה

מחברת זו מדגימה את תבנית Agentic RAG (ייצור עם חיפושים משולבים) באמצעות Microsoft Agent Framework.

**דרישות מוקדמות:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — נקודת הקצה של שירות Azure AI Search שלך
- `AZURE_SEARCH_API_KEY` — מפתח ה-API של Azure AI Search שלך
- פריסת Azure OpenAI שהוגדרה דרך משתני סביבה
- התחברות ל-Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## מהו Agentic RAG?

RAG המסורתי עוקב אחרי תהליך קבוע: משחזר מסמכים, ואז מייצר תשובה. **Agentic RAG** מתקרב יותר בכך שנותן לסוכן אוטונומיה להחליט **מתי** ו**איך** לשחזר מידע.

עם Agentic RAG, הסוכן יכול:
- **להחליט** האם יש צורך בשחזור לפני מענה על שאלה
- **לבחור** איזה מקור מידע או כלי לשאול
- **להעריך** את התוצאות שהושגו ולבצע שחזורים נוספים אם הניסיון הראשון לא מספק
- **לשלב** מידע משלב שחזור מרובים לתשובה מגובשת

זה גורם לסוכן להיות גמיש ומדויק יותר בהשוואה לתהליך השחזור-ואז-הייצור הסטטי.


## יצירת כלי חיפוש

ב-Agentic RAG, מקורות נתונים חיצוניים עטופים ככלים שהסוכן יכול להפעיל לפי דרישה. זה מאפשר לסוכן להתייחס לאחזור כפעולה נוספת שהוא יכול לבצע, במקום שלב חובה.

להלן אנו מגדירים בסיס ידע בתחום הטיולים ומציגים אותו ככלי שהסוכן יכול לקרוא לו כדי לחפש מידע על יעד.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## בניית סוכן RAG

כעת אנו יוצרים סוכן שמונחה **לתמיד לשלוף מידע לפני מענה**. הסוכן משתמש בכלי `search_travel_knowledge` כדי לעגן את תגובותיו בבסיס הידע במקום להסתמך על נתוני האימון שלו.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## אחזור איטרטיבי — תבנית יוצר-בודק

יתרון מרכזי של Agentic RAG הוא **אחזור איטרטיבי**. הסוכן יכול לבצע מספר סבבי חיפוש על מנת לוודא, לחדד או להרחיב את הממצאים הראשוניים שלו — בדומה לזרימת עבודה של "יוצר-בודק":

1. **שלב היוצר**: הסוכן מושך מידע ראשוני ומנסח תשובה.
2. **שלב הבודק**: הסוכן מבצע אחזור נוסף כדי לאמת פרטים או למלא פערים.

להלן, הסוכן נשאל שאלה שדורשת השוואה בין מספר יעדים, מה שדוחף אותו לבצע מספר חיפושים.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## סיכום

במהלך שיעור זה למדת כיצד לבנות מערכת **Agentic RAG** באמצעות Microsoft Agent Framework:

- **Agentic RAG** מאפשר לסוכנים להחליט באופן אוטונומי מתי לשלוף מידע, מה שהופך את השליפה לדינמית במקום קבועה.
- **כלים כמקורות נתונים**: מאגרי ידע חיצוניים (כמו Azure AI Search) נארזים ככלים שהסוכן יכול להפעיל.
- **שליפה איטרטיבית**: תבנית ה-maker-checker מאפשרת לסוכן לבצע מספר סבבי שליפה — חיפוש, אימות וטיוב — לפני הפקת תשובה סופית.

בייצור, תחליף את `TRAVEL_KNOWLEDGE_BASE` בזיכרון במערכת אינדקס ממשית של Azure AI Search כדי לטפל בשליפת מסמכי נסיעות בקנה מידה גדול.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
